# 15.5 Arena 竞技场评估与 Elo 评分 (Arena Evaluation & Elo Rating)

> 🕐 预估学习时间：30分钟

竞技场评估（Arena Evaluation）是当前最贴近真实用户体验的LLM评估方式。通过让不同模型对同一问题生成回答，由人类或强模型作为裁判进行两两对比（pairwise comparison），最终用Elo评分系统量化各模型的相对实力。

本节介绍Arena评估的核心流程，包括对战模拟、Elo评分计算、Bradley-Terry概率模型、统计显著性分析，以及完整的评估管线实践。

## 1. 竞技场评估概述

传统基准测试使用固定题目和标准答案，难以反映模型在真实场景中的表现。竞技场评估通过人类偏好对比，更贴近实际使用体验。

### 核心思想
- **人类偏好**：让真实用户对比模型回答，选择更优者
- **真实场景**：使用用户实际提出的多样化问题
- **两两对比**：避免绝对评分的主观性，只比较相对优劣

### 代表平台
| 平台 | 特点 |
|------|------|
| Chatbot Arena (LMSYS) | 最知名的LLM竞技场，众包人类偏好 |
| MT-Bench | 多轮对话基准，使用强模型作为裁判 |
| AlpacaEval | 基于参考答案的自动评估 |

### 评估流程
1. 收集多样化用户提示
2. 随机配对两个模型生成回答
3. 裁判（人类或强模型）判断胜者或平局
4. 用Elo评分系统更新模型评分
5. 统计分析评分的置信度与显著性

In [ ]:
import torch
import numpy as np
from collections import defaultdict

torch.manual_seed(42)

class ModelProfile:
    """模型能力画像，定义模型在不同能力维度上的强度参数。"""
    def __init__(self, name, true_skill=0.0, consistency=0.3):
        self.name = name
        self.true_skill = true_skill
        self.consistency = consistency

    def sample_performance(self):
        """采样一次表现分数：真实实力加上随机波动。"""
        noise = torch.randn(1).item() * self.consistency
        return self.true_skill + noise

class ArenaSimulator:
    """竞技场对战模拟器，生成两两对战结果数据集。"""
    def __init__(self, seed=42):
        self.seed = seed
        torch.manual_seed(seed)
        np.random.seed(seed)

    def battle(self, model_a, model_b, draw_prob=0.1):
        """模拟一次两两对战，返回结果（A胜/B胜/平局）。"""
        perf_a = model_a.sample_performance()
        perf_b = model_b.sample_performance()
        if abs(perf_a - perf_b) < 0.15 and torch.rand(1).item() < draw_prob:
            return 'draw'
        return 'a_win' if perf_a > perf_b else 'b_win'

    def run_tournament(self, models, n_battles=500):
        """运行完整锦标赛，生成对战记录数据集。"""
        battles = []
        n_models = len(models)
        for _ in range(n_battles):
            i, j = np.random.choice(n_models, 2, replace=False)
            m_a, m_b = models[i], models[j]
            result = self.battle(m_a, m_b)
            battles.append({
                'model_a': m_a.name, 'model_b': m_b.name,
                'result': result,
                'skill_a': m_a.true_skill, 'skill_b': m_b.true_skill,
            })
        return battles

    def summarize_battles(self, battles):
        """汇总对战统计信息。"""
        stats = defaultdict(lambda: {'wins': 0, 'losses': 0, 'draws': 0})
        for b in battles:
            if b['result'] == 'a_win':
                stats[b['model_a']]['wins'] += 1
                stats[b['model_b']]['losses'] += 1
            elif b['result'] == 'b_win':
                stats[b['model_b']]['wins'] += 1
                stats[b['model_a']]['losses'] += 1
            else:
                stats[b['model_a']]['draws'] += 1
                stats[b['model_b']]['draws'] += 1
        return dict(stats)

simulator = ArenaSimulator(seed=42)
models = [
    ModelProfile('ModelA', true_skill=1.5, consistency=0.4),
    ModelProfile('ModelB', true_skill=1.0, consistency=0.4),
    ModelProfile('ModelC', true_skill=0.5, consistency=0.4),
    ModelProfile('ModelD', true_skill=0.0, consistency=0.4),
    ModelProfile('ModelE', true_skill=-0.5, consistency=0.4),
]

battles = simulator.run_tournament(models, n_battles=500)
stats = simulator.summarize_battles(battles)

print('=== Arena Battle Simulation ===')
print(f'Total battles: {len(battles)}')
print(f'Models: {len(models)}')
print('\n--- Battle Statistics ---')
for name in sorted(stats.keys()):
    s = stats[name]
    total = s['wins'] + s['losses'] + s['draws']
    win_rate = s['wins'] / total if total > 0 else 0
    w, l, d = s['wins'], s['losses'], s['draws']
    print(f'  {name}: W={w}, L={l}, D={d}, '
          f'win_rate={win_rate:.2%}')

print(f'\nKey: Arena simulation generates pairwise battle data from model strength profiles.')
print(f'True skill drives win rates; consistency controls performance variance.')

## 2. Elo 评分系统

Elo评分系统由物理学家Arpad Elo提出，最初用于国际象棋，现广泛应用于各类竞技排名。它根据对战结果动态调整选手评分。

### 核心公式
**期望得分**（基于评分差）：
$$E_A = \frac{1}{1 + 10^{(R_B - R_A)/400}}$$

**评分更新**：
$$R_A' = R_A + K \cdot (S_A - E_A)$$

其中：
- $R_A, R_B$：模型A和B的当前评分
- $E_A$：模型A的期望得分（1为胜，0.5为平，0为负）
- $S_A$：实际得分
- $K$：K因子，控制单次更新幅度

### K因子选择
| K值 | 特点 | 适用场景 |
|-----|------|---------|
| K=10 | 更新保守，评分稳定 | 成熟赛事、大量对局 |
| K=32 | 平衡更新与稳定 | 通用场景 |
| K=64 | 更新激进，快速反映 | 新模型、少对局 |

In [ ]:
class EloRatingSystem:
    """Elo评分系统，根据对战结果动态更新模型评分。"""
    def __init__(self, k_factor=32, initial_rating=1000):
        self.k_factor = k_factor
        self.initial_rating = initial_rating
        self.ratings = defaultdict(lambda: float(initial_rating))
        self.history = defaultdict(list)

    def expected_score(self, rating_a, rating_b):
        """计算模型A对模型B的期望得分。"""
        return 1.0 / (1.0 + 10 ** ((rating_b - rating_a) / 400.0))

    def update(self, model_a, model_b, result):
        """根据对战结果更新两个模型的评分。

        result: 'a_win', 'b_win', or 'draw'
        """
        ra = self.ratings[model_a]
        rb = self.ratings[model_b]
        ea = self.expected_score(ra, rb)
        eb = 1.0 - ea
        if result == 'a_win':
            sa, sb = 1.0, 0.0
        elif result == 'b_win':
            sa, sb = 0.0, 1.0
        else:
            sa, sb = 0.5, 0.5
        self.ratings[model_a] = ra + self.k_factor * (sa - ea)
        self.ratings[model_b] = rb + self.k_factor * (sb - eb)
        self.history[model_a].append(self.ratings[model_a])
        self.history[model_b].append(self.ratings[model_b])

    def fit(self, battles):
        """按顺序处理所有对战记录。"""
        for b in battles:
            self.update(b['model_a'], b['model_b'], b['result'])

    def leaderboard(self):
        """返回按评分降序排列的排行榜。"""
        return sorted(self.ratings.items(), key=lambda x: -x[1])

print('=== Elo Rating System ===')
elo = EloRatingSystem(k_factor=32, initial_rating=1000)
elo.fit(battles)

print('\n--- Final Rankings (K=32) ---')
for rank, (name, rating) in enumerate(elo.leaderboard(), 1):
    print(f'  #{rank} {name}: {rating:.1f}')

# 对比不同K因子
print('\n--- K-Factor Comparison ---')
for k in [10, 32, 64]:
    elo_k = EloRatingSystem(k_factor=k, initial_rating=1000)
    elo_k.fit(battles)
    top = elo_k.leaderboard()[0]
    bottom = elo_k.leaderboard()[-1]
    spread = top[1] - bottom[1]
    print(f'  K={k:2d}: top={top[0]}({top[1]:.1f}), '
          f'bottom={bottom[0]}({bottom[1]:.1f}), spread={spread:.1f}')

# 验证评分与真实实力的相关性
true_skills = {m.name: m.true_skill for m in models}
elo_ratings = dict(elo.leaderboard())
names = sorted(true_skills.keys())
ts = np.array([true_skills[n] for n in names])
rs = np.array([elo_ratings[n] for n in names])
corr = np.corrcoef(ts, rs)[0, 1]
print(f'\nCorrelation(true_skill, elo_rating) = {corr:.4f}')

print(f'\nKey: Elo ratings converge to reflect true model strength after enough battles.')
print(f'K-factor trades off responsiveness vs stability; higher K amplifies rating spread.')

## 3. Bradley-Terry 模型

Bradley-Terry模型是Elo评分的概率论基础。它假设每个模型有一个潜在能力参数$\beta$，模型A胜过模型B的概率为：

$$P(A > B) = \frac{e^{\beta_A}}{e^{\beta_A} + e^{\beta_B}} = \sigma(\beta_A - \beta_B)$$

其中$\sigma$是sigmoid函数。

### 与Elo的关系
- Elo评分$R$与Bradley-Terry参数$\beta$的关系：$\beta = R \cdot \ln(10) / 400$
- Elo的期望得分公式就是Bradley-Terry模型的概率预测

### 最大似然估计（MLE）
给定对战数据集，通过最大化对数似然估计各模型的$\beta$参数：
$$\mathcal{L} = \sum_{(A,B) \in \text{wins}} \log P(A > B)$$

使用梯度上升法求解最优参数。Bradley-Terry模型能从全局对战数据中联合估计所有模型的能力，比Elo的顺序更新更稳定。

In [ ]:
class BradleyTerryModel:
    """Bradley-Terry模型，通过最大似然估计拟合模型能力参数。"""
    def __init__(self, n_models, lr=0.05, n_iters=500, seed=42):
        self.n_models = n_models
        self.lr = lr
        self.n_iters = n_iters
        torch.manual_seed(seed)
        self.beta = torch.zeros(n_models, requires_grad=False)

    def win_prob(self, beta_a, beta_b):
        """计算A胜B的概率（sigmoid形式）。"""
        return torch.sigmoid(beta_a - beta_b)

    def fit(self, pairs, model_to_idx):
        """使用梯度上升最大化对数似然。

        pairs: list of (winner_idx, loser_idx)
        """
        winners = torch.tensor([p[0] for p in pairs], dtype=torch.long)
        losers = torch.tensor([p[1] for p in pairs], dtype=torch.long)
        for it in range(self.n_iters):
            beta_w = self.beta[winners]
            beta_l = self.beta[losers]
            probs = torch.sigmoid(beta_w - beta_l)
            # 对数似然的梯度：dL/dbeta_w = 1 - prob, dL/dbeta_l = -(1 - prob)
            grad = (1.0 - probs)
            grad_w = torch.zeros_like(self.beta)
            grad_l = torch.zeros_like(self.beta)
            grad_w.scatter_add_(0, winners, grad)
            grad_l.scatter_add_(0, losers, -grad)
            self.beta += self.lr * (grad_w + grad_l)
            # 中心化：减去均值，消除平移不确定性
            self.beta -= self.beta.mean()
        return self

    def to_elo_rating(self, scale=400.0 / np.log(10), base=1000):
        """将Bradley-Terry参数转换为Elo评分。"""
        return self.beta * scale + base

# 准备对战数据（仅使用非平局对战）
model_names = [m.name for m in models]
model_to_idx = {n: i for i, n in enumerate(model_names)}
pairs = []
for b in battles:
    if b['result'] == 'a_win':
        pairs.append((model_to_idx[b['model_a']], model_to_idx[b['model_b']]))
    elif b['result'] == 'b_win':
        pairs.append((model_to_idx[b['model_b']], model_to_idx[b['model_a']]))

print('=== Bradley-Terry Model ===')
print(f'Effective pairs (non-draw): {len(pairs)}')

bt = BradleyTerryModel(n_models=len(models), lr=0.05, n_iters=500, seed=42)
bt.fit(pairs, model_to_idx)
bt_ratings = bt.to_elo_rating().tolist()

print('\n--- Bradley-Terry vs Elo Comparison ---')
elo_dict = dict(elo.leaderboard())
true_skills = {m.name: m.true_skill for m in models}
header = '  Model      TrueSkill        Elo    BT->Elo'
print(header)
for i, name in enumerate(model_names):
    ts_val = true_skills[name]
    elo_val = elo_dict[name]
    bt_val = bt_ratings[i]
    print(f'  {name:10s} {ts_val:>10.3f} {elo_val:>10.1f} {bt_val:>10.1f}')

# 相关性对比
ts_arr = np.array([true_skills[n] for n in model_names])
elo_arr = np.array([elo_dict[n] for n in model_names])
bt_arr = np.array(bt_ratings)
corr_elo = np.corrcoef(ts_arr, elo_arr)[0, 1]
corr_bt = np.corrcoef(ts_arr, bt_arr)[0, 1]
corr_eb = np.corrcoef(elo_arr, bt_arr)[0, 1]
print(f'\n  Corr(true, Elo)    = {corr_elo:.4f}')
print(f'  Corr(true, BT)     = {corr_bt:.4f}')
print(f'  Corr(Elo, BT)      = {corr_eb:.4f}')

print(f'\nKey: Bradley-Terry jointly estimates all model strengths via MLE.')
print(f'It often correlates better with true skill than sequential Elo updates.')

## 4. 统计显著性

竞技场评分基于有限对战样本，存在统计噪声。需要通过置信区间和显著性检验判断评分差异是否可信。

### Bootstrap重采样
1. 从原始对战数据有放回采样得到一个bootstrap样本
2. 在该样本上重新计算Elo评分
3. 重复N次，得到评分的分布
4. 取分布的2.5%和97.5%分位数作为95%置信区间

### 显著性检验
- **配对Bootstrap**：检验两个模型评分差是否显著大于0
- **重叠区间**：若置信区间重叠，差异可能不显著
- **样本量影响**：对战越多，置信区间越窄，结论越可靠

In [ ]:
class ArenaStatistics:
    """竞技场统计分析：Bootstrap置信区间与显著性检验。"""
    def __init__(self, k_factor=32, initial_rating=1000, seed=42):
        self.k_factor = k_factor
        self.initial_rating = initial_rating
        torch.manual_seed(seed)
        np.random.seed(seed)

    def compute_ratings(self, battles):
        """在给定对战数据上计算Elo评分。"""
        ratings = defaultdict(lambda: float(self.initial_rating))
        for b in battles:
            ra = ratings[b['model_a']]
            rb = ratings[b['model_b']]
            ea = 1.0 / (1.0 + 10 ** ((rb - ra) / 400.0))
            if b['result'] == 'a_win':
                sa = 1.0
            elif b['result'] == 'b_win':
                sa = 0.0
            else:
                sa = 0.5
            ratings[b['model_a']] = ra + self.k_factor * (sa - ea)
            ratings[b['model_b']] = rb + self.k_factor * ((1 - sa) - (1 - ea))
        return dict(ratings)

    def bootstrap_ci(self, battles, n_bootstrap=200, confidence=0.95):
        """使用Bootstrap计算各模型评分的置信区间。"""
        n = len(battles)
        boot_ratings = defaultdict(list)
        for _ in range(n_bootstrap):
            idx = np.random.randint(0, n, size=n)
            sample = [battles[i] for i in idx]
            ratings = self.compute_ratings(sample)
            for name, r in ratings.items():
                boot_ratings[name].append(r)
        alpha = (1 - confidence) / 2
        ci = {}
        for name, rs in boot_ratings.items():
            arr = np.array(rs)
            ci[name] = {
                'mean': float(arr.mean()),
                'std': float(arr.std()),
                'lower': float(np.percentile(arr, alpha * 100)),
                'upper': float(np.percentile(arr, (1 - alpha) * 100)),
            }
        return ci

    def significance_test(self, battles, model_x, model_y, n_bootstrap=200):
        """检验model_x评分是否显著高于model_y。"""
        n = len(battles)
        diffs = []
        for _ in range(n_bootstrap):
            idx = np.random.randint(0, n, size=n)
            sample = [battles[i] for i in idx]
            ratings = self.compute_ratings(sample)
            diffs.append(ratings[model_x] - ratings[model_y])
        diffs = np.array(diffs)
        p_value = float((diffs <= 0).mean())
        return {
            'mean_diff': float(diffs.mean()),
            'std_diff': float(diffs.std()),
            'ci_lower': float(np.percentile(diffs, 2.5)),
            'ci_upper': float(np.percentile(diffs, 97.5)),
            'p_value': p_value,
            'significant': p_value < 0.05,
        }

print('=== Arena Statistical Analysis ===')
stats_analyzer = ArenaStatistics(k_factor=32, seed=42)
ci = stats_analyzer.bootstrap_ci(battles, n_bootstrap=200, confidence=0.95)

print('\n--- 95% Confidence Intervals ---')
sorted_ci = sorted(ci.items(), key=lambda x: -x[1]['mean'])
for name, c in sorted_ci:
    mean_v = c['mean']
    lower_v = c['lower']
    upper_v = c['upper']
    width = upper_v - lower_v
    print(f'  {name:10s}: {mean_v:7.1f} [{lower_v:7.1f}, {upper_v:7.1f}] '
          f'width={width:.1f}')

# 显著性检验：相邻排名模型
print('\n--- Pairwise Significance Tests ---')
for i in range(len(sorted_ci) - 1):
    m_x = sorted_ci[i][0]
    m_y = sorted_ci[i + 1][0]
    test = stats_analyzer.significance_test(battles, m_x, m_y, n_bootstrap=200)
    sig = 'YES' if test['significant'] else 'NO'
    diff_v = test['mean_diff']
    p_v = test['p_value']
    print(f'  {m_x} > {m_y}: diff={diff_v:.1f}, '
          f'p={p_v:.4f}, significant={sig}')

print(f'\nKey: Bootstrap CIs quantify rating uncertainty from finite battle samples.')
print(f'Significance tests prevent over-interpreting small rating differences.')

## 5. 实践：Arena 评估流程

完整的Arena评估流程需要综合考虑对战设计、提示多样性和裁判偏差。

### 对战设计要点
- **提示多样性**：覆盖编码、数学、写作、推理等多类别
- **随机配对**：避免选择偏差，确保每对模型有足够对局
- **样本量**：每个模型至少100+对局才能得到稳定评分

### 裁判偏差缓解
| 偏差类型 | 表现 | 缓解方法 |
|---------|------|---------|
| 位置偏差 | 偏好先出现的回答 | 随机化回答顺序 |
| 长度偏差 | 偏好更长回答 | 长度归一化 |
| 自我偏好 | 偏好同源模型 | 多裁判交叉验证 |
| 风格偏差 | 偏好特定格式 | 多样化提示集 |

### 评估管线
1. 收集多样化提示（按类别分层采样）
2. 随机配对模型生成对战
3. 裁判判定胜负（含位置随机化）
4. 计算Elo评分与置信区间
5. 输出最终排行榜

In [ ]:
class ArenaPipeline:
    """端到端Arena评估管线：生成对战、计算评分、统计分析。"""
    def __init__(self, seed=42):
        self.seed = seed
        torch.manual_seed(seed)
        np.random.seed(seed)
        self.simulator = ArenaSimulator(seed=seed)
        self.elo_system = EloRatingSystem(k_factor=32, initial_rating=1000)
        self.stat_analyzer = ArenaStatistics(k_factor=32, seed=seed)

    def generate_models(self, n=8):
        """生成n个模型，真实能力从高到低分布。"""
        skills = np.linspace(2.0, -1.5, n)
        names = [f'Model{i}' for i in range(n)]
        return [ModelProfile(name=n, true_skill=float(s), consistency=0.45)
                for n, s in zip(names, skills)]

    def generate_prompts(self, n=200):
        """生成多样化提示，按类别分层。"""
        categories = ['coding', 'math', 'writing', 'reasoning', 'knowledge']
        weights = np.array([0.25, 0.20, 0.20, 0.20, 0.15])
        counts = np.random.multinomial(n, weights)
        prompts = []
        for cat, cnt in zip(categories, counts):
            for i in range(cnt):
                prompts.append({'category': cat, 'prompt_id': f'{cat}_{i}'})
        return prompts

    def run(self, n_models=8, n_battles=1000):
        """运行完整评估管线。"""
        models = self.generate_models(n_models)
        prompts = self.generate_prompts(n=200)
        battles = self.simulator.run_tournament(models, n_battles=n_battles)
        self.elo_system = EloRatingSystem(k_factor=32, initial_rating=1000)
        self.elo_system.fit(battles)
        ci = self.stat_analyzer.bootstrap_ci(battles, n_bootstrap=200, confidence=0.95)
        return {
            'models': models,
            'prompts': prompts,
            'battles': battles,
            'elo': self.elo_system.leaderboard(),
            'ci': ci,
        }

    def print_leaderboard(self, result):
        """打印带置信区间的最终排行榜。"""
        models = {m.name: m for m in result['models']}
        ci = result['ci']
        header = 'Rank Model      TrueSkill      Elo             95% CI   Width'
        print(header)
        for rank, (name, rating) in enumerate(result['elo'], 1):
            c = ci[name]
            ts = models[name].true_skill
            lower_v = c['lower']
            upper_v = c['upper']
            width = upper_v - lower_v
            ci_str = f'[{lower_v:.0f}, {upper_v:.0f}]'
            print(f'{rank:>4d} {name:10s} {ts:>10.3f} {rating:>8.1f} '
                  f'{ci_str:>22s} {width:>7.1f}')

print('=== Full Arena Evaluation Pipeline ===')
pipeline = ArenaPipeline(seed=42)
result = pipeline.run(n_models=8, n_battles=1000)

n_models = len(result['models'])
n_prompts = len(result['prompts'])
n_battles = len(result['battles'])
print(f'Models: {n_models}')
print(f'Prompts: {n_prompts}')
print(f'Battles: {n_battles}')

# 提示类别分布
from collections import Counter
cat_counts = Counter(p['category'] for p in result['prompts'])
print('\n--- Prompt Category Distribution ---')
for cat, cnt in sorted(cat_counts.items()):
    print(f'  {cat:10s}: {cnt}')

print('\n--- Final Leaderboard with 95% CI ---')
pipeline.print_leaderboard(result)

# 评估评分与真实实力的相关性
models = {m.name: m for m in result['models']}
ts_arr = np.array([models[n].true_skill for n, _ in result['elo']])
elo_arr = np.array([r for _, r in result['elo']])
corr = np.corrcoef(ts_arr, elo_arr)[0, 1]
print(f'\nCorrelation(true_skill, elo) = {corr:.4f}')

print(f'\nKey: Full pipeline integrates battle generation, Elo rating, and bootstrap CIs.')
print(f'Diverse prompts and position randomization ensure fair, reliable evaluation.')

## 📝 课后思考题

1. Arena竞技场评估相比传统基准测试有哪些优势？又存在什么局限性？
2. Elo评分系统中K因子的选择如何影响评分结果？在什么情况下应该使用较大的K值？
3. Bradley-Terry模型与Elo评分系统有什么联系与区别？为什么MLE估计可能比顺序更新更稳定？
4. 在实际Arena评估中，如何平衡评估的可靠性与成本？Bootstrap置信区间如何帮助决策？

---
> 本节涵盖了15.5 Arena竞技场评估与Elo评分的核心概念与代码实现。建议结合实际项目需求，选择合适的评估策略，并通过统计分析确保结论的可靠性。